In [ ]:
import sys
from pathlib import Path

# Assume the kernel cwd is this notebook's folder (neighbor of setup_fiftyone_dev_env.py).
sys.path.insert(0, str(Path.cwd()))

from setup_fiftyone_dev_env import setup

setup()

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

import fiftyone as fo

TS_NAME = "sdk_notebook_bed_temperatures"
CSV_PATH = str(Path.cwd() / "temperatures.csv")

In [ ]:
if fo.TimeSeries.exists(TS_NAME):
    fo.TimeSeries.delete(TS_NAME)

ts = fo.TimeSeries.from_csv(
    name=TS_NAME,
    filepath=CSV_PATH,
    timestamp_col="timestamp",
)
timestamps, values, channel_names = ts.to_numpy()

print(f"  Channels: {channel_names}")
print(f"  Time span: {timestamps[0]:.1f}s → {timestamps[-1]:.1f}s")

In [ ]:
means = values.mean(axis=0)
stds = values.std(axis=0)
print("Per-channel mean (°C) and std:")
for name, m, s in zip(channel_names, means, stds):
    print(f"  {name:14s}  mean={m:.2f}  std={s:.3f}")

spread = values.max(axis=1) - values.min(axis=1)
print(
    f"\nCorner temperature spread (max−min per timestep): "
    f"mean={spread.mean():.3f}°C  max={spread.max():.3f}°C"
)

In [ ]:
corr = np.corrcoef(values.T)
print("Channel correlation matrix (rows/cols = " + ", ".join(channel_names) + "):")
with np.printoptions(precision=3, suppress=True):
    print(corr)

## Plot (0–60s window, all channels)

Uses `fo.TimeSeries.query` then `to_plotly()` for an inline interactive figure.

In [ ]:
window = ts.query(start=0.0, end=60.0)
n = len(window[channel_names[0]])
print(f"Plotting 0–60s window: {n} samples per channel")

fig = go.Figure(window.to_plotly())
fig.update_layout(title=f"{TS_NAME} (0–60s) — SDK demo")
fig.show()